# Editroom ComfyUI Colab Server (Cloudflare Tunnel / Tailscale)

Run this notebook on Google Colab with a GPU runtime (T4 / L4 / A100), then copy the printed tunnel URL into **Editroom > Settings > Integrations > ComfyUI**.

In [ ]:
#@title 1. Install ComfyUI & Dependencies
!nvidia-smi
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt


In [ ]:
#@title 2. Start ComfyUI + Cloudflare Tunnel (Recommended) or Tailscale
import subprocess, threading, time, re, urllib.request

TUNNEL_MODE = "cloudflare" #@param ["cloudflare", "tailscale"]
TAILSCALE_AUTHKEY = "" #@param {type:"string"}

if TUNNEL_MODE == "cloudflare":
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1 || true
elif TUNNEL_MODE == "tailscale":
    !curl -fsSL https://tailscale.com/install.sh | sh >/dev/null 2>&1
    if TAILSCALE_AUTHKEY:
        !tailscaled --tun=userspace-networking &
        time.sleep(2)
        !tailscale up --authkey={TAILSCALE_AUTHKEY} --hostname=editroom-colab
        !tailscale ip -4

def wait_for_comfy():
    while True:
        try:
            with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as r:
                if r.status == 200:
                    break
        except Exception:
            pass
        time.sleep(1)
    if TUNNEL_MODE == "cloudflare":
        p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m:
                print("\n==> COPY THIS URL INTO EDITROOM INTEGRATIONS:", m.group(0), flush=True)
    else:
        print("\n==> Tailscale ready! Use http://<tailscale-ip>:8188 in Editroom Integrations.", flush=True)

threading.Thread(target=wait_for_comfy, daemon=True).start()
!python main.py --listen 127.0.0.1 --port 8188 --dont-print-server
